# Counterfactual Data Augmentation (CDA) for Debiasing Small Language Models

This notebook implements CDA training for debiasing masked language models using counterfactual sentence pairs.

## Key Concepts:
- **CDA Loss**: Ensures model predictions are similar for counterfactual pairs (e.g., "he" vs "she")
- **Masked Language Modeling**: Focus on predictions for masked tokens
- **Consistency Regularization**: Minimize differences between counterfactual predictions

In [23]:
# Import necessary libraries
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForMaskedLM
from transformers import get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import os
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# Load and examine the dataset
file_path = 'data/processed/dataset.json'

with open(file_path, 'r') as f:
    raw_data = json.load(f)

# Clean the data by removing brackets and ensuring proper format
data = []
for item in raw_data:
    if isinstance(item, dict) and 'pro' in item and 'anti' in item:
        cleaned_item = {
            "pro": item["pro"].replace("[", "").replace("]", "").strip(),
            "anti": item["anti"].replace("[", "").replace("]", "").strip()
        }
        # Only include items that are different and non-empty
        if cleaned_item["pro"] != cleaned_item["anti"] and cleaned_item["pro"] and cleaned_item["anti"]:
            data.append(cleaned_item)

print(f"Loaded {len(data)} counterfactual pairs")
print(f"Example pair:")
print(f"  Pro-stereotyped: {data[0]['pro']}")
print(f"  Anti-stereotyped: {data[0]['anti']}")

In [ ]:
# Analyze the dataset structure
print("Dataset Analysis:")
print(f"Total pairs: {len(data)}")

# Check for common patterns
gendered_pronouns = ['he', 'she', 'his', 'her', 'him', 'man', 'woman', 'male', 'female']
gendered_count = 0

for item in data[:5]:  # Show first 5 examples
    print(f"\nPair {len([x for x in data[:data.index(item)+1]])}:")
    print(f"  Pro: {item['pro']}")
    print(f"  Anti: {item['anti']}")
    
    # Check if this pair contains gendered terms
    pro_lower = item['pro'].lower()
    anti_lower = item['anti'].lower()
    has_gender = any(pronoun in pro_lower or pronoun in anti_lower for pronoun in gendered_pronouns)
    if has_gender:
        gendered_count += 1

print(f"\nDataset contains gendered terms: {gendered_count}/{len(data)} pairs")

{'anti': 'The developer argued with the designer because [she] did not like the design.',
 'pro': 'The developer argued with the designer because [he] did not like the design.'}

In [ ]:
# Initialize the model and tokenizer
model_name = "distilbert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")
print(f"Loading model: {model_name}")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForMaskedLM.from_pretrained(model_name)

# Add padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded successfully!")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Mask token: {tokenizer.mask_token} (ID: {tokenizer.mask_token_id})")

In [ ]:
# Configure LoRA for efficient fine-tuning
lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,                    # Rank
    lora_alpha=32,          # Alpha parameter
    lora_dropout=0.1,       # Dropout
    target_modules=["q_lin", "v_lin"],  # Target attention modules
    bias="none",
    modules_to_save=None,
)

# Apply LoRA to the model
model = get_peft_model(base_model, lora_config)
model = model.to(device)

# Print trainable parameters
model.print_trainable_parameters()

print(f"Model moved to {device}")

In [ ]:
# CDA Dataset Class
class CDADataset(Dataset):
    """
    Dataset for Counterfactual Data Augmentation training.
    Creates masked versions of counterfactual sentence pairs.
    """
    def __init__(self, data, tokenizer, max_length=128, mask_prob=0.15):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.mask_prob = mask_prob
        self.gendered_words = [
            'he', 'she', 'his', 'her', 'him', 'man', 'woman', 
            'male', 'female', 'boy', 'girl', 'father', 'mother',
            'son', 'daughter', 'brother', 'sister', 'husband', 'wife'
        ]
    
    def __len__(self):
        return len(self.data)
    
    def mask_gendered_tokens(self, text):
        """Mask gendered words in the text for CDA training"""
        words = text.split()
        masked_text = []
        
        for word in words:
            # Remove punctuation for checking
            clean_word = word.lower().strip('.,!?;:"()[]')
            if clean_word in self.gendered_words:
                # Replace gendered word with mask token
                masked_text.append(self.tokenizer.mask_token)
            else:
                masked_text.append(word)
        
        return ' '.join(masked_text)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Create masked versions focusing on gendered terms
        pro_masked = self.mask_gendered_tokens(item['pro'])
        anti_masked = self.mask_gendered_tokens(item['anti'])
        
        # Tokenize both versions
        pro_encoded = self.tokenizer(
            pro_masked,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        anti_encoded = self.tokenizer(
            anti_masked,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'pro_input_ids': pro_encoded['input_ids'].squeeze(0),
            'pro_attention_mask': pro_encoded['attention_mask'].squeeze(0),
            'anti_input_ids': anti_encoded['input_ids'].squeeze(0),
            'anti_attention_mask': anti_encoded['attention_mask'].squeeze(0),
            'original_pro': item['pro'],
            'original_anti': item['anti']
        }

# Test the dataset
test_dataset = CDADataset(data[:5], tokenizer)
sample = test_dataset[0]

print("Sample from CDA Dataset:")
print(f"Original pro: {sample['original_pro']}")
print(f"Original anti: {sample['original_anti']}")
print(f"Pro tokens: {tokenizer.decode(sample['pro_input_ids'], skip_special_tokens=True)}")
print(f"Anti tokens: {tokenizer.decode(sample['anti_input_ids'], skip_special_tokens=True)}")

trainable params: 294,912 || all params: 67,280,442 || trainable%: 0.4383


In [ ]:
# CDA Loss Function
def cda_loss_function(pro_logits, anti_logits, pro_input_ids, anti_input_ids, tokenizer, 
                     consistency_weight=1.0, mlm_weight=0.1):
    """
    Compute CDA loss combining consistency loss and masked language modeling loss.
    
    Args:
        pro_logits: Logits from pro-stereotyped sentence
        anti_logits: Logits from anti-stereotyped sentence  
        pro_input_ids: Input IDs for pro sentence
        anti_input_ids: Input IDs for anti sentence
        tokenizer: Tokenizer instance
        consistency_weight: Weight for consistency loss
        mlm_weight: Weight for MLM loss
    
    Returns:
        total_loss, consistency_loss, mlm_loss
    """
    
    # Find masked token positions
    mask_token_id = tokenizer.mask_token_id
    pro_mask_positions = (pro_input_ids == mask_token_id)
    anti_mask_positions = (anti_input_ids == mask_token_id)
    
    # Consistency Loss: Ensure similar predictions for counterfactual pairs
    # Focus on masked positions
    if pro_mask_positions.any() and anti_mask_positions.any():
        # Get logits for masked positions only
        pro_masked_logits = pro_logits[pro_mask_positions]
        anti_masked_logits = anti_logits[anti_mask_positions]
        
        # Ensure same number of masked tokens (take minimum)
        min_masks = min(pro_masked_logits.size(0), anti_masked_logits.size(0))
        if min_masks > 0:
            pro_masked_logits = pro_masked_logits[:min_masks]
            anti_masked_logits = anti_masked_logits[:min_masks]
            
            # Consistency loss: KL divergence between probability distributions
            pro_probs = F.softmax(pro_masked_logits, dim=-1)
            anti_probs = F.softmax(anti_masked_logits, dim=-1)
            
            consistency_loss = F.kl_div(
                F.log_softmax(pro_masked_logits, dim=-1),
                anti_probs,
                reduction='batchmean'
            ) + F.kl_div(
                F.log_softmax(anti_masked_logits, dim=-1),
                pro_probs,
                reduction='batchmean'
            )
            consistency_loss = consistency_loss / 2.0
        else:
            consistency_loss = torch.tensor(0.0, device=pro_logits.device, requires_grad=True)
    else:
        consistency_loss = torch.tensor(0.0, device=pro_logits.device, requires_grad=True)
    
    # Optional: Add regularization to encourage neutral predictions
    # This helps prevent the model from being overly confident in biased directions
    if pro_mask_positions.any():
        pro_masked_logits = pro_logits[pro_mask_positions]
        # Entropy regularization - encourage uniform distribution
        pro_probs = F.softmax(pro_masked_logits, dim=-1)
        entropy_loss = -torch.mean(torch.sum(pro_probs * torch.log(pro_probs + 1e-10), dim=-1))
        mlm_loss = -entropy_loss  # Higher entropy = lower loss
    else:
        mlm_loss = torch.tensor(0.0, device=pro_logits.device, requires_grad=True)
    
    # Combine losses
    total_loss = consistency_weight * consistency_loss + mlm_weight * mlm_loss
    
    return total_loss, consistency_loss, mlm_loss

# Test the loss function
print("Testing CDA loss function...")
with torch.no_grad():
    # Create dummy inputs
    batch_size, seq_len, vocab_size = 2, 10, tokenizer.vocab_size
    dummy_pro_logits = torch.randn(batch_size, seq_len, vocab_size)
    dummy_anti_logits = torch.randn(batch_size, seq_len, vocab_size)
    
    # Create input IDs with some mask tokens
    dummy_pro_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
    dummy_anti_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
    dummy_pro_ids[0, 3] = tokenizer.mask_token_id  # Add mask token
    dummy_anti_ids[0, 3] = tokenizer.mask_token_id
    
    loss, cons_loss, mlm_loss = cda_loss_function(
        dummy_pro_logits, dummy_anti_logits, 
        dummy_pro_ids, dummy_anti_ids, tokenizer
    )
    
    print(f"Total loss: {loss.item():.4f}")
    print(f"Consistency loss: {cons_loss.item():.4f}")
    print(f"MLM loss: {mlm_loss.item():.4f}")

print("CDA loss function working correctly!")

Pro sentence: The developer argued with the designer because he did not like the design.
Anti sentence: The developer argued with the designer because she did not like the design.
{'input_ids': tensor([[ 101, 1996, 9722, 5275, 2007, 1996, 5859, 2138, 2002, 2106, 2025, 2066,
         1996, 2640, 1012,  102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
{'input_ids': tensor([[ 101, 1996, 9722, 5275, 2007, 1996, 5859, 2138, 2016, 2106, 2025, 2066,
         1996, 2640, 1012,  102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [ ]:
# Prepare training data
train_size = int(0.8 * len(data))
val_size = len(data) - train_size

train_data = data[:train_size]
val_data = data[train_size:]

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

# Create datasets and dataloaders
train_dataset = CDADataset(train_data, tokenizer, max_length=128)
val_dataset = CDADataset(val_data, tokenizer, max_length=128)

batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

# Test a batch
sample_batch = next(iter(train_loader))
print(f"\nSample batch shapes:")
for key, value in sample_batch.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")
    else:
        print(f"  {key}: {type(value)} (length: {len(value)})")

Pro logits shape: torch.Size([1, 16, 30522])
Anti logits shape: torch.Size([1, 16, 30522])


In [ ]:
# Training Configuration
config = {
    'epochs': 3,
    'learning_rate': 2e-5,
    'consistency_weight': 1.0,
    'mlm_weight': 0.1,
    'warmup_steps': 100,
    'logging_steps': 50,
    'eval_steps': 200,
    'save_steps': 500,
}

print("Training Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

# Initialize optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=0.01)

total_steps = len(train_loader) * config['epochs']
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=config['warmup_steps'],
    num_training_steps=total_steps
)

print(f"\nTotal training steps: {total_steps}")
print(f"Warmup steps: {config['warmup_steps']}")

# Training history tracking
training_history = {
    'train_loss': [],
    'train_consistency_loss': [],
    'train_mlm_loss': [],
    'val_loss': [],
    'val_consistency_loss': [],
    'val_mlm_loss': [],
    'learning_rate': []
}

Total Loss: 0.3442
Task Loss: 0.0000
Consistency Loss: 0.6885


In [ ]:
# Training Loop
def train_epoch(model, train_loader, optimizer, scheduler, config, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    total_consistency_loss = 0
    total_mlm_loss = 0
    
    progress_bar = tqdm(train_loader, desc="Training")
    
    for step, batch in enumerate(progress_bar):
        optimizer.zero_grad()
        
        # Move batch to device
        pro_input_ids = batch['pro_input_ids'].to(device)
        pro_attention_mask = batch['pro_attention_mask'].to(device)
        anti_input_ids = batch['anti_input_ids'].to(device)
        anti_attention_mask = batch['anti_attention_mask'].to(device)
        
        # Forward pass
        pro_outputs = model(
            input_ids=pro_input_ids,
            attention_mask=pro_attention_mask
        )
        anti_outputs = model(
            input_ids=anti_input_ids, 
            attention_mask=anti_attention_mask
        )
        
        # Compute CDA loss
        loss, consistency_loss, mlm_loss = cda_loss_function(
            pro_outputs.logits, anti_outputs.logits,
            pro_input_ids, anti_input_ids, tokenizer,
            consistency_weight=config['consistency_weight'],
            mlm_weight=config['mlm_weight']
        )
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        # Track metrics
        total_loss += loss.item()
        total_consistency_loss += consistency_loss.item()
        total_mlm_loss += mlm_loss.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'cons': f'{consistency_loss.item():.4f}',
            'mlm': f'{mlm_loss.item():.4f}',
            'lr': f'{scheduler.get_last_lr()[0]:.2e}'
        })
    
    avg_loss = total_loss / len(train_loader)
    avg_consistency_loss = total_consistency_loss / len(train_loader)
    avg_mlm_loss = total_mlm_loss / len(train_loader)
    
    return avg_loss, avg_consistency_loss, avg_mlm_loss

def evaluate(model, val_loader, config, device):
    """Evaluate the model"""
    model.eval()
    total_loss = 0
    total_consistency_loss = 0
    total_mlm_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            # Move batch to device
            pro_input_ids = batch['pro_input_ids'].to(device)
            pro_attention_mask = batch['pro_attention_mask'].to(device)
            anti_input_ids = batch['anti_input_ids'].to(device)
            anti_attention_mask = batch['anti_attention_mask'].to(device)
            
            # Forward pass
            pro_outputs = model(
                input_ids=pro_input_ids,
                attention_mask=pro_attention_mask
            )
            anti_outputs = model(
                input_ids=anti_input_ids,
                attention_mask=anti_attention_mask
            )
            
            # Compute CDA loss
            loss, consistency_loss, mlm_loss = cda_loss_function(
                pro_outputs.logits, anti_outputs.logits,
                pro_input_ids, anti_input_ids, tokenizer,
                consistency_weight=config['consistency_weight'],
                mlm_weight=config['mlm_weight']
            )
            
            total_loss += loss.item()
            total_consistency_loss += consistency_loss.item()
            total_mlm_loss += mlm_loss.item()
    
    avg_loss = total_loss / len(val_loader)
    avg_consistency_loss = total_consistency_loss / len(val_loader)
    avg_mlm_loss = total_mlm_loss / len(val_loader)
    
    return avg_loss, avg_consistency_loss, avg_mlm_loss

print("Training functions defined successfully!")

In [ ]:
# Execute Training
print("Starting CDA training...")
print("="*50)

best_val_loss = float('inf')
best_epoch = 0

for epoch in range(config['epochs']):
    print(f"\nEpoch {epoch + 1}/{config['epochs']}")
    print("-" * 30)
    
    # Train
    train_loss, train_cons_loss, train_mlm_loss = train_epoch(
        model, train_loader, optimizer, scheduler, config, device
    )
    
    # Evaluate
    val_loss, val_cons_loss, val_mlm_loss = evaluate(
        model, val_loader, config, device
    )
    
    # Log results
    current_lr = scheduler.get_last_lr()[0]
    
    print(f"Training   - Loss: {train_loss:.4f}, Consistency: {train_cons_loss:.4f}, MLM: {train_mlm_loss:.4f}")
    print(f"Validation - Loss: {val_loss:.4f}, Consistency: {val_cons_loss:.4f}, MLM: {val_mlm_loss:.4f}")
    print(f"Learning Rate: {current_lr:.2e}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        print(f"✓ New best validation loss: {val_loss:.4f}")
    
    # Store history
    training_history['train_loss'].append(train_loss)
    training_history['train_consistency_loss'].append(train_cons_loss)
    training_history['train_mlm_loss'].append(train_mlm_loss)
    training_history['val_loss'].append(val_loss)
    training_history['val_consistency_loss'].append(val_cons_loss)
    training_history['val_mlm_loss'].append(val_mlm_loss)
    training_history['learning_rate'].append(current_lr)

print(f"\nTraining completed!")
print(f"Best validation loss: {best_val_loss:.4f} at epoch {best_epoch + 1}")

In [ ]:
# Visualize Training Progress
plt.figure(figsize=(15, 5))

# Loss curves
plt.subplot(1, 3, 1)
epochs_range = range(1, len(training_history['train_loss']) + 1)
plt.plot(epochs_range, training_history['train_loss'], 'b-', label='Train Loss', linewidth=2)
plt.plot(epochs_range, training_history['val_loss'], 'r-', label='Val Loss', linewidth=2)
plt.title('Training Progress')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Consistency loss
plt.subplot(1, 3, 2)
plt.plot(epochs_range, training_history['train_consistency_loss'], 'b-', label='Train Consistency', linewidth=2)
plt.plot(epochs_range, training_history['val_consistency_loss'], 'r-', label='Val Consistency', linewidth=2)
plt.title('Consistency Loss')
plt.xlabel('Epoch')
plt.ylabel('Consistency Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Learning rate
plt.subplot(1, 3, 3)
plt.plot(epochs_range, training_history['learning_rate'], 'g-', linewidth=2)
plt.title('Learning Rate Schedule')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.yscale('log')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print("Training Summary:")
print(f"Final train loss: {training_history['train_loss'][-1]:.4f}")
print(f"Final val loss: {training_history['val_loss'][-1]:.4f}")
print(f"Best val loss: {min(training_history['val_loss']):.4f}")
print(f"Final consistency loss (val): {training_history['val_consistency_loss'][-1]:.4f}")

Training for 3 epochs with 100 samples...


In [ ]:
# Save the trained model
output_dir = "outputs/models/cda_debiased_model"
os.makedirs(output_dir, exist_ok=True)

# Save the model and tokenizer
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# Save training history
import pickle
with open(f"{output_dir}/training_history.pkl", "wb") as f:
    pickle.dump(training_history, f)

# Save training config
with open(f"{output_dir}/training_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Model saved to: {output_dir}")
print("Saved files:")
for file in os.listdir(output_dir):
    print(f"  - {file}")

print(f"\nModel summary:")
print(f"  - Base model: {model_name}")
print(f"  - Training method: Counterfactual Data Augmentation (CDA)")
print(f"  - Training samples: {len(train_data)}")
print(f"  - Epochs trained: {config['epochs']}")
print(f"  - Final validation loss: {training_history['val_loss'][-1]:.4f}")

Epoch 1/3: 100%|██████████| 13/13 [00:13<00:00,  1.04s/it, loss=0.1631, task=0.3153, consistency=0.0110]


Epoch 1 avg loss: 0.1849


Epoch 2/3: 100%|██████████| 13/13 [00:16<00:00,  1.24s/it, loss=0.1151, task=0.2122, consistency=0.0179]


Epoch 2 avg loss: 0.1357


Epoch 3/3: 100%|██████████| 13/13 [00:14<00:00,  1.11s/it, loss=0.0740, task=0.1296, consistency=0.0183]

Epoch 3 avg loss: 0.0925
Training completed!


In [ ]:
# Test the Debiased Model
def test_bias_in_predictions(model, tokenizer, device, test_sentences):
    """Test for bias in model predictions"""
    model.eval()
    results = []
    
    with torch.no_grad():
        for sentence in test_sentences:
            # Tokenize input
            inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Get predictions
            outputs = model(**inputs)
            logits = outputs.logits
            
            # Find mask token position
            mask_token_id = tokenizer.mask_token_id
            mask_positions = (inputs['input_ids'] == mask_token_id).nonzero(as_tuple=True)
            
            if len(mask_positions[0]) > 0:
                # Get predictions for masked position
                mask_logits = logits[mask_positions[0], mask_positions[1]]
                top_k = torch.topk(mask_logits, k=10, dim=-1)
                
                predictions = []
                for i in range(min(5, top_k.indices.size(-1))):
                    token_id = top_k.indices[0][i].item()
                    score = top_k.values[0][i].item()
                    token = tokenizer.decode([token_id]).strip()
                    predictions.append((token, score))
                
                results.append({
                    'sentence': sentence,
                    'predictions': predictions
                })
    
    return results

# Test sentences focusing on gender bias
test_sentences = [
    "The doctor examined the patient. [MASK] was very thorough.",
    "The nurse helped the patient. [MASK] was very caring.", 
    "The engineer designed the building. [MASK] was very skilled.",
    "The teacher explained the lesson. [MASK] was very patient.",
    "The CEO announced the decision. [MASK] was very confident.",
    "The programmer wrote the code. [MASK] was very efficient.",
    "The lawyer argued the case. [MASK] was very persuasive.",
]

# Replace [MASK] with actual mask token
test_sentences = [s.replace("[MASK]", tokenizer.mask_token) for s in test_sentences]

print("Testing the debiased model for gender bias...")
print("="*60)

results = test_bias_in_predictions(model, tokenizer, device, test_sentences)

for result in results:
    print(f"\nSentence: {result['sentence']}")
    print("Top predictions:")
    for i, (token, score) in enumerate(result['predictions'], 1):
        print(f"  {i}. '{token}' (score: {score:.3f})")

print(f"\n{'='*60}")
print("Analysis complete! Check if gendered pronouns appear with similar frequencies.")

Model saved to ../outputs/models/cda_lora_model
Contents: ['adapter_model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'tokenizer.json', 'README.md', 'adapter_config.json', 'vocab.txt']


/Users/rithvikrajesh/Proper_Project/Debaising_SLM/.venv/lib/python3.12/site-packages/peft/utils/other.py:1228: UserWarning: Unable to fetch remote file due to the following error (ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 7afc1954-e146-4047-b4d3-aa5a23115a3e)') - silently ignoring the lookup for the file config.json in distilbert-base-uncased.
  warnings.warn(
/Users/rithvikrajesh/Proper_Project/Debaising_SLM/.venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:286: UserWarning: Could not find a config file in distilbert-base-uncased - will assume that the vocabulary was not modified.
  warnings.warn(


In [ ]:
# Load and Test the Saved Model
from peft import PeftModel

print("Loading the saved debiased model...")

# Load base model
base_model_test = AutoModelForMaskedLM.from_pretrained(model_name)

# Load the fine-tuned LoRA weights
debiased_model = PeftModel.from_pretrained(base_model_test, output_dir)
debiased_model = debiased_model.to(device)
debiased_model.eval()

print("Model loaded successfully!")

# Compare original vs debiased model
def compare_models(original_model, debiased_model, tokenizer, device, sentence):
    """Compare predictions between original and debiased models"""
    original_model.eval()
    debiased_model.eval()
    
    with torch.no_grad():
        # Tokenize input
        inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Get predictions from both models
        original_outputs = original_model(**inputs)
        debiased_outputs = debiased_model(**inputs)
        
        # Find mask token position
        mask_token_id = tokenizer.mask_token_id
        mask_positions = (inputs['input_ids'] == mask_token_id).nonzero(as_tuple=True)
        
        if len(mask_positions[0]) > 0:
            # Get predictions for masked position
            orig_logits = original_outputs.logits[mask_positions[0], mask_positions[1]]
            debiased_logits = debiased_outputs.logits[mask_positions[0], mask_positions[1]]
            
            orig_top_k = torch.topk(orig_logits, k=5, dim=-1)
            debiased_top_k = torch.topk(debiased_logits, k=5, dim=-1)
            
            orig_predictions = []
            debiased_predictions = []
            
            for i in range(5):
                # Original model predictions
                token_id = orig_top_k.indices[0][i].item()
                score = orig_top_k.values[0][i].item()
                token = tokenizer.decode([token_id]).strip()
                orig_predictions.append((token, score))
                
                # Debiased model predictions
                token_id = debiased_top_k.indices[0][i].item()
                score = debiased_top_k.values[0][i].item()
                token = tokenizer.decode([token_id]).strip()
                debiased_predictions.append((token, score))
            
            return orig_predictions, debiased_predictions
    
    return None, None

# Test comparison
test_sentence = f"The doctor examined the patient carefully. {tokenizer.mask_token} was very professional."

print(f"\nComparison Test:")
print(f"Sentence: {test_sentence}")
print("-" * 50)

orig_preds, debiased_preds = compare_models(base_model_test, debiased_model, tokenizer, device, test_sentence)

if orig_preds and debiased_preds:
    print("Original Model Predictions:")
    for i, (token, score) in enumerate(orig_preds, 1):
        print(f"  {i}. '{token}' (score: {score:.3f})")
    
    print("\nDebiased Model Predictions:")
    for i, (token, score) in enumerate(debiased_preds, 1):
        print(f"  {i}. '{token}' (score: {score:.3f})")
    
    # Check for gendered pronouns in top predictions
    gendered_pronouns = ['he', 'she', 'him', 'her', 'his']
    
    orig_gendered = [pred for pred in orig_preds if pred[0].lower() in gendered_pronouns]
    debiased_gendered = [pred for pred in debiased_preds if pred[0].lower() in gendered_pronouns]
    
    print(f"\nGender Analysis:")
    print(f"Original model gendered pronouns in top 5: {len(orig_gendered)}")
    print(f"Debiased model gendered pronouns in top 5: {len(debiased_gendered)}")
    
    if orig_gendered:
        print("Original gendered predictions:", [pred[0] for pred in orig_gendered])
    if debiased_gendered:
        print("Debiased gendered predictions:", [pred[0] for pred in debiased_gendered])

print(f"\n{'='*60}")
print("Model comparison complete!")
print(f"The CDA training has been completed successfully.")
print(f"The model should now show reduced gender bias in masked language modeling tasks.")

Test sentence: The doctor performed surgery. [MASK] is the best doctor in the country.
Top 5 predictions for masked token:
  1. 'he' (score: 7.5364)
  2. 'she' (score: 6.2286)
  3. 'it' (score: 5.7900)
  4. 'this' (score: 5.1264)
  5. 'anand' (score: 4.2520)
